# 04 — A/B Testing & Model Routing Demo

This notebook demonstrates:
- Champion/Challenger model lifecycle
- Rollback via API
- Register new challenger via API
- Traffic distribution analysis

**Prerequisites**: Docker services running

In [2]:
import random
from collections import Counter

import httpx

API_URL = "http://localhost:8001"

## 1. Current Champion Model

In [3]:
resp = httpx.get(f"{API_URL}/models/credit-risk")
model = resp.json()
print(f"Champion: v{model['version']} ({model['status']})")
print(f"Metrics: {model.get('metadata', {}).get('metrics', {})}")

Champion: vv1 (development)
Metrics: {'accuracy': 0.85}


## 2. Rollback Active Challengers

In [4]:
resp = httpx.post(f"{API_URL}/models/rollback", json={"model_id": "credit-risk"})
result = resp.json()
print(f"Champion preserved: {result['champion']}")
print(f"Archived challengers: {result['archived_challengers']}")

Champion preserved: v1
Archived challengers: ['v1773941701']


## 3. Register a New Challenger

In [5]:
resp = httpx.post(
    f"{API_URL}/models/register",
    json={
        "model_id": "credit-risk",
        "version": "v-notebook-demo",
        "uri": "local:///models/credit_risk/demo/model.onnx",
        "framework": "onnx",
        "stage": "challenger",
        "metrics": {"accuracy": 0.92, "f1": 0.91, "auc": 0.95},
    },
)
result = resp.json()
print(f"Registered: {result['version']} as {result['stage']}")
print(f"Status: {result['status']}")

Registered: v-notebook-demo as challenger
Status: registered


## 4. Simulate Traffic Distribution

In production, the routing strategy (A/B, Canary, Shadow) distributes
traffic between champion and challenger. Here we simulate the distribution.

In [6]:
NUM_REQUESTS = 1000
CANARY_PERCENTAGE = 10  # 10% to challenger

routing = Counter()
for _ in range(NUM_REQUESTS):
    if random.random() < CANARY_PERCENTAGE / 100:
        routing["challenger"] += 1
    else:
        routing["champion"] += 1

print(f"Traffic Distribution (n={NUM_REQUESTS}):")
for model_type, count in routing.most_common():
    pct = count / NUM_REQUESTS * 100
    bar = "\u2588" * int(pct / 2)
    print(f"  {model_type:12s} {bar} {pct:.1f}% ({count})")

Traffic Distribution (n=1000):
  champion     ████████████████████████████████████████████ 89.6% (896)
  challenger   █████ 10.4% (104)


## 5. Cleanup — Archive Demo Challenger

In [7]:
resp = httpx.post(f"{API_URL}/models/rollback", json={"model_id": "credit-risk"})
result = resp.json()
print(f"Cleaned up: archived {result['archived_challengers']}")
print(f"Champion still serving: {result['champion']}")

Cleaned up: archived []
Champion still serving: v1
